# CIFAR-10 split and dataset figures

Purpose: build the one stratified split used by Q1–Q4, and look at the data before any model is trained.

Question: shared setup (before Q1). Later notebooks import `load_experiment_arrays` from `src/common/data.py` instead of splitting again.

How to run: from the repository root, with the environment in `README.md`,

` .venv/bin/python scripts/prepare_data.py `

This notebook calls the same functions. It does not contain a second split.

## Rules this notebook is enforcing

- Official CIFAR-10 training pool: 50,000 images. We keep 45,000 to train and 5,000 to validate.
- The split is stratified: 4,500 / 500 in every class. Seed `26003` is the digits of roll `m26csa003` in order.
- Validation is only for later model choices. The official 10,000 test images are not used to compute the saved mean and standard deviation.
- If `splits/split_seed26003.npz` already exists, it is reloaded and checked. It is not redrawn.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.common.data import SEED, load_experiment_arrays, prepare_dataset
from src.common.utils import get_device, set_seed
from src.common.viz import write_dataset_figures

set_seed(SEED)
print("seed", SEED)
print("device that later training will see:", get_device())
print("project root", ROOT)

In [ ]:
prepared = prepare_dataset(ROOT)
report = prepared["report"]
print("split file", report["split_status"])
print("train / val / test", report["train_size"], report["val_size"], report["test_size"])
print("overlap", report["overlap"])
print("train counts", report["train_counts"])
print("val counts  ", report["val_counts"])
print("test counts ", report["test_counts"])
print("mean", [round(v, 6) for v in report["mean_rgb"]])
print("std ", [round(v, 6) for v in report["std_rgb"]])
print("train index sha256", report["train_idx_sha256"])

## What the checks mean

Equal counts in every class mean the validation score is not dominated by one class appearing more often. Zero overlap, and a union that covers indices `0 ... 49999`, means we did not drop or double-count any official training image. Test counts are read from test labels only, to confirm the download. The mean and standard deviation above were reduced from the training images only.

In [ ]:
write_dataset_figures(prepared, ROOT / "figures", ROOT / "results" / "phase1")
print("wrote figures to", ROOT / "figures")
print((ROOT / "figures" / "dataset_description.md").read_text())

In [ ]:
from IPython.display import Image, display

for name in ["class_samples.png", "class_distribution.png", "mean_images.png", "pixel_statistics.png"]:
    print(name)
    display(Image(filename=str(ROOT / "figures" / name)))

## Loader used by later questions

`load_experiment_arrays` returns float32 images in `[0, 1]` with shape `(N, 3, 32, 32)`. Train and validation follow the saved indices. The test tensor is the full official test set. Images are not normalised yet: classifiers should call `apply_train_normalisation`, while a reconstruction model can stay in `[0, 1]`. The index order is sorted, so it is not an epoch order.

In [ ]:
arrays = load_experiment_arrays(ROOT, device="cpu")
for key in ["train_images", "val_images", "test_images"]:
    tensor = arrays[key]
    print(f"{key:12} shape={tuple(tensor.shape)} dtype={tensor.dtype} min={tensor.min():.3f} max={tensor.max():.3f}")